# Anthropometrically Personalized Squat Coach — real-time, with skeleton tracking

**What this notebook gives you**

| Your requirement | Where it lives |
|---|---|
| **Adaptive assessment** — form judged against *your* proportions, not a universal rule | balance model + personalized network (sections 1–2) |
| **Anthropometric calibration** — upload a standing side-view photo (or several, or a 2–3 s clip), or calibrate with the camera → femur/torso, shank/femur, leg/torso ratios | live app, **Step 1** |
| **Personalized ML pipeline** — joint angles **+** static body ratios → classifier | section 2 (trained here, exported in section 5, runs in the browser) |
| **Individualized feedback & form score** — geometry-aware messages and a 0–100 score per rep, live **or on an uploaded video of a past session** | live app, **Step 2** |
| **Fairness audit** — personalized vs generic baselines across body types, with confidence intervals | sections 3–4 |

**Pipeline**

```
Webcam ─► MediaPipe Pose (33 landmarks, in-browser, ~30 fps) ─► skeleton drawn live
      └► 3D world landmarks ─► joint angles (knee, hip, torso, shin, depth)
Calibration (standing, side-on, 3 s) ─► femur/torso · shank/femur · leg/torso ratios
[angles + ratios] ─► personalized MLP ─► form score 0-100      (+ balance-model corrective cues)
[angles only]     ─► generic MLP      ─► baseline score        (+ fixed-threshold rule baseline)
```

**Why the live part runs in the browser:** Colab runs on a remote machine without access to your camera. Streaming frames through Python gives ~1 fps. Running MediaPipe in the browser (WebAssembly/GPU) gives real-time skeleton tracking; the model you train here is exported to JSON and executed in the page.

### Quick start
1. *Runtime ▸ Run all* (sections 1–5 take a few minutes, mostly the sensitivity sweep).
2. In the last cell the app asks for **Step 1 · Personalize**: **upload a standing, side-on photo** of the person (relaxed, full body incl. feet; several photos or a 2–3 s clip make the ratios more robust). No photo? Use the live-camera calibration button instead.
3. **Step 2 · Analyse** — pick one:
   * **Upload a squat video** to check a past session. It plays with the skeleton and the real-time feature panel; you can pause, scrub, change speed, and get a rep log + summary at the end. No separate photo? Use *"Use the next 3 s of this video as my calibration"* while the person is standing.
   * **Start live camera** (side-on, camera at hip height, 2–3 m away).
4. Watch the **real-time input features** panel (shoulder / hip / knee / ankle coordinates, `knee_angle`, `torso_lean`, `hip_depth_ratio`, and the body ratios that go into the model). The skeleton turns **green** when you match *your* geometry; the dashed **cyan line** shows where *your* torso should be at that depth.

### Honest limitations (please keep these in your write-up)
* The classifier is trained on a **physics-based synthetic population** because no labelled real dataset ships with the notebook. That makes the audit a demonstration of the *method*. The real evidence requires coach-labelled reps from real people — the live app has a built-in labelling/CSV export for exactly that (section 7).
* The balance model is a 2D side-view simplification (bar/centre of mass over mid-foot). It ignores bar position, stance width and spine curvature.
* Single-camera pose has noise; the sensitivity analysis in section 5 shows how conclusions depend on it.

## 1 · Setup, one shared config, and the biomechanics model

Every threshold lives in one `CFG` dict that is also injected into the browser app, so training/audit (Python) and live inference (JavaScript) cannot drift apart.

**Balance model (what makes the system "geometry-aware").** In a side view, with the shoulders (bar / centre of mass) kept over the mid-foot:

`thigh·cos(φ) − shank·sin(α) + foot_offset = sin(τ)` (all lengths in torso-lengths)

so the torso lean **τ** a person *needs* depends on *their* thigh and shank lengths relative to their torso (φ = thigh elevation, 0 = parallel; α = shin lean).

In [ ]:
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# ONE place for every constant. The same values are injected into the browser
# app, so Python (training/audit) and JavaScript (live) can never drift apart.
# ---------------------------------------------------------------------------
CFG = dict(
    foot_offset      = 0.14,   # mid-foot sits this far ahead of the ankle (in torso lengths)
    depth_phi_ok     = 5.0,    # deg: thigh elevation <= this => hip crease at/below knee ("depth reached")
    torso_tol        = 8.0,    # deg: allowed gap between actual and balance-derived torso lean
    alpha_range      = (26.0, 46.0),  # deg: typical shank lean at depth across ankle mobilities (used for target ranges)
    alpha_max        = 52.0,   # deg: shank lean above this = knees travelling excessively far forward
    generic_knee_max = 90.0,   # "one-size-fits-all" rule: knee angle at the bottom <= 90 deg ...
    generic_torso_max= None,   # ... and torso lean <= X deg: filled in below from an AVERAGE build
    calib_seconds    = 3,      # standing calibration length
    rep_start_knee   = 140.0,  # knee angle below which a rep has started
    rep_end_knee     = 160.0,  # knee angle above which the rep has ended
    min_rep_knee     = 125.0,  # a rep must reach at least this much knee flexion to be counted
    calib_max_frames = 30,     # max frames sampled from an uploaded calibration clip
    calib_clip_max_s = 8.0,    # only the first N seconds of an uploaded calibration clip are used
    calib_max_px     = 1280,   # uploaded photos/frames are downscaled to this longest side before pose detection
)

KIN_FEATS   = ["knee", "hip", "torso", "shank", "phi"]               # real-time kinematics (deg)
RATIO_FEATS = ["femur_torso", "shank_femur", "leg_torso"]            # static anthropometric ratios
ALL_FEATS   = KIN_FEATS + RATIO_FEATS

# ---------------------------------------------------------------------------
# Planar squat biomechanics (side view). Units: torso length = 1.
#   ankle at origin, y up.  alpha = shank lean from vertical,
#   phi = thigh elevation above horizontal (0 = parallel, <0 = below parallel),
#   tau = torso lean from vertical.
#   Balance condition: shoulders (bar / centre of mass) stay over the mid-foot
#       thigh*cos(phi) - shank*sin(alpha) + foot_offset = sin(tau) * 1
# => the torso lean a person NEEDS depends on THEIR femur/shank/torso lengths.
# ---------------------------------------------------------------------------
def torso_needed(t_r, s_r, alpha, phi, foot=CFG["foot_offset"]):
    """Balance-derived torso lean (deg). t_r = thigh/torso, s_r = shank/torso."""
    a, p = np.radians(alpha), np.radians(phi)
    return np.degrees(np.arcsin(np.clip(t_r * np.cos(p) - s_r * np.sin(a) + foot, -1, 1)))

# A one-size-fits-all rule is, in practice, tuned on an "average" body: the balanced torso lean of an
# average-proportioned person at parallel depth, plus the usual tolerance.
AVG_BUILD = dict(t_r=0.85, s_r=0.85, alpha=36.0)
CFG["generic_torso_max"] = round(float(torso_needed(AVG_BUILD["t_r"], AVG_BUILD["s_r"], AVG_BUILD["alpha"], 0.0))
                                 + CFG["torso_tol"], 1)
print("Generic rule thresholds -> knee <= %.0f deg, torso lean <= %.1f deg, hip at/below knee level"
      % (CFG["generic_knee_max"], CFG["generic_torso_max"]))

def _angle(u, v):
    cosv = (u * v).sum(-1) / (np.linalg.norm(u, axis=-1) * np.linalg.norm(v, axis=-1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cosv, -1, 1)))

def pose_angles(t_r, s_r, alpha, phi, tau):
    """Knee angle (hip-knee-ankle) and hip angle (shoulder-hip-knee) implied by the geometry."""
    a, p, ta = np.radians(alpha), np.radians(phi), np.radians(tau)
    K = np.asarray(s_r)[..., None] * np.stack([np.sin(a), np.cos(a)], -1)
    H = K + np.asarray(t_r)[..., None] * np.stack([-np.cos(p), np.sin(p)], -1)
    S = H + np.stack([np.sin(ta), np.cos(ta)], -1)
    return _angle(H - K, -K), _angle(S - H, K - H)

# ---------------------------------------------------------------------------
# Synthetic population + squat-rep simulator.
# IMPORTANT: this is a stand-in so the whole pipeline runs end to end today.
# Replace it with coach-labelled reps recorded in the live app (see "REAL DATA").
# ---------------------------------------------------------------------------
def simulate_dataset(n_people=800, reps_per_person=25, seed=0, noise_deg=5.0):
    r = np.random.default_rng(seed)
    out = []
    for pid in range(n_people):
        ft = float(np.clip(r.normal(0.85, 0.08), 0.62, 1.10))   # true thigh / torso
        sf = float(np.clip(r.normal(1.00, 0.07), 0.80, 1.20))   # true shank / thigh
        t_r, s_r = ft, ft * sf
        alpha_nat = r.uniform(*CFG["alpha_range"])               # personal ankle mobility
        bt, bs = r.normal(0, 0.03, 2)                           # calibration error (one per person/session)
        t_m, s_m = t_r * (1 + bt), s_r * (1 + bs)               # what the camera calibration "sees"

        n = reps_per_person
        alpha = np.clip(alpha_nat + r.normal(0, 3, n), 10, 62)
        phi = np.where(r.random(n) < 0.75, r.uniform(-22, 3, n), r.uniform(7, 35, n))
        need = torso_needed(t_r, s_r, alpha, phi)
        sign = r.choice([-1, 1], n)
        tau = np.where(r.random(n) < 0.65, need + r.normal(0, 4, n), need + sign * r.uniform(11, 26, n))
        tau = np.clip(tau, 0, 85)

        # ground-truth "coach" label from the person's own geometry (+ coach tolerance jitter + 3% disagreement)
        tol = r.uniform(CFG["torso_tol"] - 2, CFG["torso_tol"] + 2, n)
        good = (phi <= CFG["depth_phi_ok"]) & (np.abs(tau - need) <= tol) & (alpha <= CFG["alpha_max"])
        flip = r.random(n) < 0.03
        good = np.where(flip, ~good, good)

        knee, hip = pose_angles(np.full(n, t_r), np.full(n, s_r), alpha, phi, tau)
        noise = lambda: r.normal(0, noise_deg, n)               # pose-estimation error on every angle (deg)
        out.append(pd.DataFrame(dict(
            person_id=pid, rep_id=np.arange(n),
            knee=knee + noise(), hip=hip + noise(), torso=tau + noise(), shank=alpha + noise(), phi=phi + noise(),
            femur_torso=t_m, shank_femur=s_m / t_m, leg_torso=t_m + s_m,
            label=good.astype(int), tau_needed=need)))
    return pd.concat(out, ignore_index=True)

In [ ]:
# Why a single threshold cannot be right for everyone: the torso lean a person NEEDS at parallel depth
ft = np.linspace(0.62, 1.10, 60)                         # femur / torso ratio across the population
mid = torso_needed(ft, ft * 1.0, 36.0, 0.0)              # balanced lean for typical ankle mobility
lo  = torso_needed(ft, ft * 1.0, CFG["alpha_range"][1], 0.0)
hi  = torso_needed(ft, ft * 1.0, CFG["alpha_range"][0], 0.0)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.fill_between(ft, lo - CFG["torso_tol"], hi + CFG["torso_tol"], color="#16a34a", alpha=.18, label="Balanced torso lean (range over ankle mobility, +/- tolerance)")
ax.plot(ft, mid, color="#16a34a", lw=2, label="Balanced lean, typical ankle mobility")
ax.axhline(CFG["generic_torso_max"], color="#dc2626", ls="--", lw=2, label=f"One-size-fits-all rule: lean <= {CFG['generic_torso_max']} deg")
ax.axvline(0.85, color="grey", ls=":", lw=1); ax.text(0.855, 3, "average build", fontsize=8, color="grey")
ax.set_xlabel("Femur-to-torso ratio (thigh length / torso length)"); ax.set_ylabel("Torso lean at parallel depth (deg from vertical)")
ax.set_title("Correct form looks geometrically different for different bodies", fontsize=11)
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
print("People with long femurs are pushed past the fixed limit while squatting with perfectly balanced form.")

## 2 · Data, split, and the three systems under comparison

* **Generic rule** – fixed thresholds (what most fitness apps do), tuned for an *average* build.
* **Generic ML** – same network, **joint angles only**.
* **Personalized ML** – same network, **joint angles + anthropometric ratios**.

Train/test split is **by person** (nobody appears in both), so the audit measures generalisation to unseen bodies.  
Set `USE_REAL_DATA = True` to train on your own collected CSV instead of the simulator.

In [ ]:
USE_REAL_DATA = False   # <- set True after you have collected coach-labelled reps with the live app (see REAL DATA cell)

if USE_REAL_DATA:
    from google.colab import files
    print("Upload the CSV(s) exported from the live app (columns: person_id, rep_id, " + ", ".join(ALL_FEATS) + ", label)")
    up = files.upload()
    df = pd.concat([pd.read_csv(f) for f in up], ignore_index=True)
    df = df.dropna(subset=ALL_FEATS + ["label"])
    df["label"] = df["label"].astype(int)
    print("Real reps loaded:", len(df), "| people:", df.person_id.nunique())
else:
    df = simulate_dataset(n_people=800, reps_per_person=25, seed=0)
    print("Synthetic reps:", len(df), "| people:", df.person_id.nunique(), "| share of good reps: %.2f" % df.label.mean())

# --- person-level split: nobody appears in both train and test ---------------------------------
people = df.person_id.unique()
rng.shuffle(people)
cut = int(0.7 * len(people))
train_people, test_people = set(people[:cut]), set(people[cut:])
tr, te = df[df.person_id.isin(train_people)].reset_index(drop=True), df[df.person_id.isin(test_people)].reset_index(drop=True)
print(f"train reps {len(tr)} ({len(train_people)} people) | test reps {len(te)} ({len(test_people)} people)")

# --- three systems under comparison -------------------------------------------------------------
def fit_mlp(X, y, seed=0):
    sc = StandardScaler().fit(X)
    clf = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=1e-3, max_iter=400,
                        early_stopping=True, n_iter_no_change=15, random_state=seed)
    clf.fit(sc.transform(X), y)
    return sc, clf

def proba(sc, clf, X):
    return clf.predict_proba(sc.transform(X))[:, 1]

# 1) GENERIC RULE: the fixed thresholds most fitness apps use for everyone
def rule_predict(d):
    return ((d.knee <= CFG["generic_knee_max"]) & (d.torso <= CFG["generic_torso_max"]) &
            (d.phi <= CFG["depth_phi_ok"])).astype(int).values

# 2) GENERIC ML: same network, kinematics only (no body information)
gen_sc, gen_clf = fit_mlp(tr[KIN_FEATS].values, tr.label.values, seed=1)
# 3) PERSONALIZED ML: same network, kinematics + anthropometric ratios
per_sc, per_clf = fit_mlp(tr[ALL_FEATS].values, tr.label.values, seed=1)

te["p_rule"] = rule_predict(te).astype(float)
te["p_gen"]  = proba(gen_sc, gen_clf, te[KIN_FEATS].values)
te["p_per"]  = proba(per_sc, per_clf, te[ALL_FEATS].values)

METHODS = {"Generic rule (fixed thresholds)": "p_rule",
           "Generic ML (angles only)":        "p_gen",
           "Personalized ML (angles + ratios)":"p_per"}

summary = []
for name, col in METHODS.items():
    pred = (te[col] >= 0.5).astype(int)
    summary.append(dict(method=name, accuracy=accuracy_score(te.label, pred), F1=f1_score(te.label, pred)))
summary = pd.DataFrame(summary).set_index("method").round(3)
display(summary) if "display" in globals() else print(summary)

## 3 · Fairness audit across body types

Test people are grouped into tertiles of femur-to-torso ratio (short / average / long femur) and of leg-to-torso ratio.

* **FRR (false-rejection rate)** – correct form wrongly flagged as incorrect. This is the "algorithmic unfairness" in your problem statement.
* **FAR (false-acceptance rate)** – faulty form wrongly passed.
* **Gap** – max − min of a rate across groups. A fair system has a small gap.
* Confidence intervals come from a **bootstrap over people** (not reps), because reps from one person are not independent.

In [ ]:
GROUP_NAMES = ["Low", "Mid", "High"]

def person_groups(d, col):
    """Assign every PERSON to a tertile of a body-proportion ratio (median of their calibrations)."""
    pm = d.groupby("person_id")[col].median()
    q = pd.qcut(pm.rank(method="first"), 3, labels=GROUP_NAMES)
    return d.person_id.map(q).astype(str)

def rates(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    good, bad = (y == 1), (y == 0)
    frr = np.mean(pred[good] == 0) if good.any() else np.nan     # correct form wrongly flagged "incorrect"
    far = np.mean(pred[bad] == 1) if bad.any() else np.nan       # faulty form wrongly accepted
    return frr, far, np.mean(pred == y)

def audit(d, group_col, title):
    d = d.copy()
    d["grp"] = person_groups(d, group_col)
    rows = []
    for name, col in METHODS.items():
        pred = (d[col] >= 0.5).astype(int)
        for g in GROUP_NAMES:
            m = d.grp == g
            frr, far, acc = rates(d.label[m], pred[m])
            rows.append(dict(method=name, group=g, n_reps=int(m.sum()), FRR=frr, FAR=far, accuracy=acc))
    t = pd.DataFrame(rows)
    print(f"\n=== Fairness audit by {title} tertile ===")
    print("FRR = correct reps wrongly marked incorrect | FAR = faulty reps wrongly accepted")
    piv = t.pivot(index="method", columns="group", values="FRR")[GROUP_NAMES].round(3)
    piv["FRR gap (max-min)"] = (piv.max(axis=1) - piv.min(axis=1)).round(3)
    print("\nFalse-rejection rate (lower = fairer to good form):")
    print(piv.to_string())
    piv2 = t.pivot(index="method", columns="group", values="accuracy")[GROUP_NAMES].round(3)
    piv2["acc gap (max-min)"] = (piv2.max(axis=1) - piv2.min(axis=1)).round(3)
    print("\nAccuracy:")
    print(piv2.to_string())
    return d, t

def bootstrap_gap(d, n_boot=500, seed=0):
    """Resample PEOPLE with replacement; CI for each method's FRR gap and for (baseline gap - personalized gap)."""
    r = np.random.default_rng(seed)
    pids = d.person_id.unique()
    gid = {g: i for i, g in enumerate(GROUP_NAMES)}
    pg = d.groupby("person_id").grp.first().map(gid).reindex(pids).values
    cnt = {}
    for name, col in METHODS.items():
        fr = ((d[col] < 0.5) & (d.label == 1)).groupby(d.person_id).sum().reindex(pids).values
        cnt[name] = fr
    ng = (d.label == 1).groupby(d.person_id).sum().reindex(pids).values
    gaps = {k: [] for k in METHODS}
    for _ in range(n_boot):
        idx = r.integers(0, len(pids), len(pids))
        den = np.bincount(pg[idx], weights=ng[idx], minlength=3)
        for name in METHODS:
            num = np.bincount(pg[idx], weights=cnt[name][idx], minlength=3)
            frr = num / np.maximum(den, 1)
            gaps[name].append(frr.max() - frr.min())
    gaps = {k: np.array(v) for k, v in gaps.items()}
    print("\nBootstrap (people resampled) FRR-gap, mean [95% CI]:")
    for name in METHODS:
        lo, hi = np.percentile(gaps[name], [2.5, 97.5])
        print(f"  {name:38s} {gaps[name].mean():.3f}  [{lo:.3f}, {hi:.3f}]")
    pers = "Personalized ML (angles + ratios)"
    for base in ["Generic rule (fixed thresholds)", "Generic ML (angles only)"]:
        diff = gaps[base] - gaps[pers]
        lo, hi = np.percentile(diff, [2.5, 97.5])
        verdict = "personalized is significantly fairer" if lo > 0 else "difference not significant"
        print(f"  gap reduction vs {base:34s} {diff.mean():+.3f}  [{lo:+.3f}, {hi:+.3f}]  -> {verdict}")

# ---- main audit: femur-to-torso ratio (the proportion that drives squat mechanics most) ----------
d1, t1 = audit(te, "femur_torso", "femur-to-torso ratio")
bootstrap_gap(d1)
# ---- second audit: leg-to-torso ratio ---------------------------------------------------------------
d2, t2 = audit(te, "leg_torso", "leg-to-torso ratio")

# ---- figure -----------------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
colors = ["#9ca3af", "#60a5fa", "#16a34a"]
for ax, metric, ttl in zip(axes, ["FRR", "FAR"],
                           ["False-rejection rate (good form flagged wrong)", "False-acceptance rate (bad form passed)"]):
    w = 0.26
    for i, name in enumerate(METHODS):
        vals = t1[t1.method == name].set_index("group").loc[GROUP_NAMES, metric].values
        ax.bar(np.arange(3) + (i - 1) * w, vals, w, label=name, color=colors[i])
    ax.set_xticks(range(3)); ax.set_xticklabels(["Short femur\n(low femur/torso)", "Average", "Long femur\n(high femur/torso)"])
    ax.set_title(ttl, fontsize=11); ax.set_ylim(0, 1); ax.grid(axis="y", alpha=.3)
axes[0].legend(fontsize=8, loc="upper left")
plt.suptitle("Fairness audit across body types (test people never seen in training)", fontsize=12)
plt.tight_layout(); plt.show()

# ---- sanity: is the personalized model really using the ratios? (shuffle ratios between people) -------
shuf = te.copy()
perm = shuf.groupby("person_id")[RATIO_FEATS].first().sample(frac=1, random_state=3)
perm.index = shuf.groupby("person_id")[RATIO_FEATS].first().index
for c in RATIO_FEATS:
    shuf[c] = shuf.person_id.map(perm[c])
acc_real = accuracy_score(te.label, (te.p_per >= .5).astype(int))
acc_shuf = accuracy_score(shuf.label, (proba(per_sc, per_clf, shuf[ALL_FEATS].values) >= .5).astype(int))
print(f"\nRatio-shuffle check: accuracy {acc_real:.3f} with the right body ratios -> {acc_shuf:.3f} with someone else's ratios")

### How to read this
* Against the **fixed-threshold rule**, personalization should cut the false-rejection gap sharply: the rule punishes long-femur people for balanced form.
* Against **generic ML (angles only)** the picture is subtler: joint angles at the bottom of a squat partly *encode* body shape, so an angles-only network can implicitly infer some of it. In the sensitivity sweep (section 4) the explicit ratios give higher accuracy at every noise level and equal-or-lower long-femur false rejections, but a significant *gap* reduction versus generic ML is **not** guaranteed. If the interval above spans zero, report that honestly — it is a finding, not a failure, and real data will settle it.
* Explicit ratios also give something an angles-only model cannot: **individual targets and corrective cues** ("your balanced torso lean here is ≈ 34°, you are at 47°").

## 4 · Sensitivity: does the benefit depend on how noisy pose estimation is?

Cleaner angles leak more body-shape information, so the extra value of explicit ratios depends on pose noise. Sweep the assumed joint-angle error. (≈ 1–2 minutes.)

In [ ]:
# Does the benefit of explicit body ratios survive different assumptions about pose-estimation error?
# (Joint angles carry some hidden information about body shape, so the cleaner the angles, the less
#  the ratios add. Real webcam pose noise is a key quantity to MEASURE in your own data.)
def quick_compare(noise_deg, n_people=700, reps=20, seed=11):
    d = simulate_dataset(n_people, reps, seed=seed, noise_deg=noise_deg)
    ppl = d.person_id.unique(); r = np.random.default_rng(seed); r.shuffle(ppl)
    tr_ = d[d.person_id.isin(ppl[:int(.7 * len(ppl))])]; te_ = d[~d.person_id.isin(ppl[:int(.7 * len(ppl))])].copy()
    g_sc, g_clf = fit_mlp(tr_[KIN_FEATS].values, tr_.label.values, 1)
    p_sc, p_clf = fit_mlp(tr_[ALL_FEATS].values, tr_.label.values, 1)
    te_["grp"] = person_groups(te_, "femur_torso")
    out = {}
    for name, pred in [("rule", rule_predict(te_)),
                       ("generic_ML", (proba(g_sc, g_clf, te_[KIN_FEATS].values) >= .5).astype(int)),
                       ("personalized_ML", (proba(p_sc, p_clf, te_[ALL_FEATS].values) >= .5).astype(int))]:
        frr = [rates(te_.label[te_.grp == g], pred[(te_.grp == g).values])[0] for g in GROUP_NAMES]
        out[name] = dict(acc=accuracy_score(te_.label, pred), FRR_long_femur=frr[2], FRR_gap=max(frr) - min(frr))
    return out

rows = []
for nd in [2.0, 5.0, 8.0, 12.0]:
    res = quick_compare(nd)
    for k, v in res.items():
        rows.append(dict(pose_noise_deg=nd, method=k, **{a: round(b, 3) for a, b in v.items()}))
sens = pd.DataFrame(rows)
print(sens.pivot(index="pose_noise_deg", columns="method", values=["acc", "FRR_gap", "FRR_long_femur"]).round(3).to_string())

## 5 · Export the trained models to the browser

The two networks (personalized: 8 inputs, generic: 5 inputs) are exported as plain weight arrays. The cell proves the export is lossless by re-running the exact browser forward pass in NumPy and comparing it with scikit-learn.

In [ ]:
def export_mlp(sc, clf, feats):
    return dict(features=feats, mean=sc.mean_.tolist(), scale=sc.scale_.tolist(),
                W=[w.tolist() for w in clf.coefs_], b=[b.tolist() for b in clf.intercepts_])

def np_forward(M, x):
    """Re-implementation of exactly what the browser will run (ReLU hidden layers, sigmoid output)."""
    a = (np.asarray(x, float) - M["mean"]) / M["scale"]
    for i, (W, b) in enumerate(zip(M["W"], M["b"])):
        a = a @ np.asarray(W) + np.asarray(b)
        if i < len(M["W"]) - 1:
            a = np.maximum(a, 0)
    return 1 / (1 + np.exp(-a[0]))

MODELS = dict(personalized=export_mlp(per_sc, per_clf, ALL_FEATS),
              generic=export_mlp(gen_sc, gen_clf, KIN_FEATS))

# parity test: the exported weights must reproduce sklearn's probabilities
chk = te.sample(300, random_state=0)
d_per = max(abs(np_forward(MODELS["personalized"], r[ALL_FEATS].values) - p) for (_, r), p in zip(chk.iterrows(), chk.p_per))
d_gen = max(abs(np_forward(MODELS["generic"], r[KIN_FEATS].values) - p) for (_, r), p in zip(chk.iterrows(), chk.p_gen))
print(f"Export parity (max |difference| vs scikit-learn): personalized {d_per:.2e}, generic {d_gen:.2e}")
assert d_per < 1e-6 and d_gen < 1e-6, "exported weights do not match the trained model!"
print("Models exported OK ->", {k: f"{len(v['mean'])} inputs" for k, v in MODELS.items()})

## 6 · ▶ Interactive coach: personalize from a photo, then analyse live or an uploaded video

* **Step 1 · Personalize** — upload standing side-view photo(s) / a short clip (processed locally in your browser; nothing is uploaded anywhere), or calibrate with the camera. The app shows the detected skeleton on your photo and your resulting ratios and personal targets.
* **Step 2 · Analyse** — *Start live camera* **or** *Upload a squat video to check*. Video mode has play/pause, speed (0.25×–1.5×), a seek bar, and a per-rep **log with timestamps** and an end-of-video summary.
* **Real-time input features** — live shoulder / hip / knee / ankle coordinates, `knee_angle`, `hip_angle`, `torso_lean` (with *your* target), `shin_lean`, `thigh_elevation`, `hip_depth_ratio`, and the three body ratios: exactly what the network receives. (`hip_depth_ratio` is shown for reference only; it assumes the same camera distance as calibration, so the model uses the more robust `thigh_elevation` instead.)
* **Per-rep form score (0–100)** with corrective cues, plus the *same rep* scored by the generic ML and the fixed rule, so the fairness difference is visible.

Tips: for uploaded videos use side-on footage with the whole body visible; MP4 (H.264) and WebM play in all browsers, and phone HEIC photos must be converted to JPG/PNG. If the camera doesn't start, allow camera access or just use the upload option; you can also run the standalone saved file locally (optional cell after the app).

In [ ]:
#@title ▶ Launch the real-time coach (skeleton tracking runs in your browser) { display-mode: "form" }
from IPython.display import HTML, display

APP_TEMPLATE = r"""<div id="sq-app" style="font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:#e5e7eb;background:#0f172a;border-radius:12px;padding:12px;max-width:1080px;box-sizing:border-box">
  <style>
    #sq-app button,#sq-app label.btn{display:inline-block;background:#2563eb;color:#fff;border:0;border-radius:8px;padding:8px 12px;font-size:13px;cursor:pointer;margin:2px 4px 2px 0}
    #sq-app button:disabled{background:#475569;cursor:not-allowed;opacity:.7}
    #sq-app button.alt{background:#334155} #sq-app button.good{background:#16a34a} #sq-app button.bad{background:#dc2626} #sq-app label.btn.orange{background:#c2410c}
    #sq-app .card{background:#1e293b;border-radius:10px;padding:10px 12px;margin-bottom:8px;font-size:13px;line-height:1.45}
    #sq-app .card h4{margin:0 0 6px 0;font-size:12px;letter-spacing:.06em;text-transform:uppercase;color:#94a3b8}
    #sq-app .big{font-size:40px;font-weight:700;line-height:1}
    #sq-app .bar{height:10px;border-radius:6px;background:#334155;overflow:hidden;margin-top:6px}
    #sq-app .bar>div{height:100%;width:0;background:#22c55e;transition:width .15s}
    #sq-app .ok{color:#4ade80} #sq-app .warn{color:#fbbf24} #sq-app .err{color:#f87171} #sq-app .mut{color:#94a3b8}
    #sq-app table{border-collapse:collapse;width:100%} #sq-app td{padding:2px 4px;font-size:13px;vertical-align:top}
    #sq-app td.k{color:#94a3b8;white-space:nowrap} #sq-app td.v{font-family:ui-monospace,Menlo,Consolas,monospace}
    #sq-app tr.sep td{border-top:1px solid #334155;padding-top:4px}
    #sq-app input,#sq-app select{background:#0f172a;color:#e5e7eb;border:1px solid #475569;border-radius:6px;padding:4px 6px;font-size:12px}
    #sq-app textarea{width:100%;height:70px;background:#0f172a;color:#cbd5e1;border:1px solid #475569;border-radius:6px;font-size:11px;box-sizing:border-box}
  </style>

  <div class="card" style="border:1px solid #c2410c">
    <h4>Step 1 &middot; Personalize (body calibration)</h4>
    <div class="mut" style="margin-bottom:8px">Upload one or more <b>standing, side-on photos</b> of the person (relaxed, full body incl. feet in frame) <b>or a short 2&ndash;3 s clip</b>. More frames make the body-ratio estimate more robust to landmark jitter. No photo? Calibrate with the live camera instead.</div>
    <label class="btn orange">&#128247; Upload standing photo(s) / short clip<input id="sq-calfile" type="file" accept="image/*,video/*" multiple style="display:none"></label>
    <button id="sq-cal" class="alt" disabled>&#127919; or calibrate with live camera (3 s)</button>
    <div id="sq-calstatus" class="warn" style="margin-top:6px">Not personalized yet &mdash; upload a standing photo to begin.</div>
  </div>

  <div class="card">
    <h4>Step 2 &middot; Analyse the movement</h4>
    <button id="sq-start">&#9654; Start live camera</button>
    <label class="btn">&#127910; Upload a squat video to check<input id="sq-vidfile" type="file" accept="video/*" style="display:none"></label>
    <button id="sq-stop" class="alt" disabled>Stop</button>
    <button id="sq-reset" class="alt">Reset session</button>
    <select id="sq-model" title="pose model"><option value="lite">Pose model: lite (fastest)</option><option value="full" selected>full (balanced)</option><option value="heavy">heavy (most accurate)</option></select>
    <div id="sq-vctl" style="display:none;margin-top:8px">
      <button id="sq-play" class="alt">Pause</button>
      Speed <select id="sq-speed"><option value="0.25">0.25&times;</option><option value="0.5">0.5&times;</option><option value="1" selected>1&times;</option><option value="1.5">1.5&times;</option></select>
      <input id="sq-seek" type="range" min="0" max="1000" value="0" style="width:min(320px,45%);vertical-align:middle">
      <span id="sq-time" class="mut">0:00 / 0:00</span>
      <button id="sq-vcal" class="alt">Use the next 3 s of this video as my calibration</button>
    </div>
  </div>

  <div style="display:flex;flex-wrap:wrap;gap:12px">
    <div style="flex:1 1 560px;min-width:300px">
      <div id="sq-stage" style="position:relative;width:100%;aspect-ratio:16/9;background:#000;border-radius:10px;overflow:hidden">
        <video id="sq-video" playsinline muted style="position:absolute;left:0;top:0;width:100%;height:100%;object-fit:contain"></video>
        <canvas id="sq-canvas" style="position:absolute;left:0;top:0;width:100%;height:100%"></canvas>
        <div id="sq-banner" style="position:absolute;left:8px;right:8px;top:8px;padding:6px 10px;border-radius:8px;background:rgba(15,23,42,.82);font-size:14px;font-weight:600">Start with Step 1: upload a standing side-on photo. Then start the camera or upload a squat video.</div>
        <div id="sq-fps" style="position:absolute;right:8px;bottom:6px;font-size:11px;color:#cbd5e1;background:rgba(15,23,42,.6);padding:2px 6px;border-radius:6px"></div>
      </div>
      <div class="card" style="margin-top:8px"><h4>Live depth</h4>
        <div class="bar"><div id="sq-depthbar"></div></div>
        <div class="mut" style="margin-top:4px" id="sq-depthtxt">&mdash;</div>
      </div>
      <div class="card"><h4>Real-time input features (what the model sees)</h4><div id="sq-feat" class="mut">Waiting for a person in view&hellip;</div></div>
    </div>

    <div style="flex:1 1 300px;min-width:280px">
      <div class="card"><h4>Your body profile</h4><div id="sq-profile" class="mut">Not calibrated yet.</div></div>
      <div class="card"><h4>Live feedback</h4><div id="sq-live" class="mut">&mdash;</div></div>
      <div class="card"><h4>Rep <span id="sq-repno">0</span> &mdash; form score</h4>
        <div style="display:flex;align-items:center;gap:14px"><div class="big" id="sq-score">&ndash;</div><div id="sq-verdict" style="flex:1"></div></div>
        <div class="bar"><div id="sq-scorebar"></div></div>
        <div id="sq-repdetail" style="margin-top:8px" class="mut">Do a squat (or play a video) to get a score.</div>
        <div class="mut" style="margin-top:6px">Session average: <b id="sq-avg">&ndash;</b></div>
      </div>
      <div class="card"><h4>Rep log</h4><div id="sq-sum" class="mut" style="margin-bottom:4px"></div><div id="sq-log" class="mut">No reps yet.</div></div>
      <div class="card"><h4>Collect labelled data (to train on real people)</h4>
        <div class="mut" style="margin-bottom:6px">After a rep, record what a coach says. Participant ID: <input id="sq-pid" value="p01" size="6"></div>
        <button id="sq-good" class="good" disabled>Last rep was GOOD</button><button id="sq-bad" class="bad" disabled>Last rep was FAULTY</button>
        <button id="sq-dl" class="alt" disabled>Download CSV</button>
        <div class="mut" style="margin:4px 0">Labelled reps: <b id="sq-nrows">0</b> (if the download is blocked, copy the text below)</div>
        <textarea id="sq-csv" readonly></textarea>
      </div>
    </div>
  </div>
</div>

<script>
(async function () {
  if (window.__sqStop) { try { window.__sqStop(); } catch (e) {} }
  const CFG = __CFG__;
  const MODELS = __MODELS__;

  // ====================================================================== PURE:BEGIN
  const D2R = Math.PI / 180, R2D = 180 / Math.PI;
  const clamp = (v, a, b) => Math.min(b, Math.max(a, v));
  const sub = (a, b) => [a.x - b.x, a.y - b.y, a.z - b.z];
  const dot = (u, v) => u[0] * v[0] + u[1] * v[1] + u[2] * v[2];
  const nrm = u => Math.hypot(u[0], u[1], u[2]);
  const median = a => { const s = [...a].sort((x, y) => x - y), m = s.length >> 1; return s.length % 2 ? s[m] : (s[m - 1] + s[m]) / 2; };

  function angleAt(a, b, c) {
    const u = sub(a, b), v = sub(c, b);
    return Math.acos(clamp(dot(u, v) / (nrm(u) * nrm(v) + 1e-9), -1, 1)) * R2D;
  }
  function fromVertical(u) { return Math.acos(clamp(Math.abs(u[1]) / (nrm(u) + 1e-9), 0, 1)) * R2D; }

  // Same definitions as the Python simulator: knee=(hip,knee,ankle), hip=(shoulder,hip,knee),
  // torso/shank = lean from vertical, phi = thigh elevation above horizontal (0 = parallel).
  // upSign = sign(hip.y - knee.y) measured while standing, so the code works whatever the y-axis direction is.
  function featuresFromWorld(sh, hip, kn, an, upSign) {
    const thighV = sub(hip, kn);
    return {
      knee: angleAt(hip, kn, an), hip: angleAt(sh, hip, kn),
      torso: fromVertical(sub(sh, hip)), shank: fromVertical(sub(kn, an)),
      phi: Math.asin(clamp(upSign * thighV[1] / (nrm(thighV) + 1e-9), -1, 1)) * R2D,
      thighLen: nrm(thighV), shankLen: nrm(sub(kn, an)), torsoLen: nrm(sub(sh, hip))
    };
  }
  function profileFromLengths(thigh, shank, torso, upSign) {
    return { t_r: thigh / torso, s_r: shank / torso, femur_torso: thigh / torso,
             shank_femur: shank / thigh, leg_torso: (thigh + shank) / torso, upSign: upSign };
  }
  // Balance model: torso lean this person needs so the shoulders stay over the mid-foot.
  function torsoNeeded(P, alpha, phi) {
    return Math.asin(clamp(P.t_r * Math.cos(phi * D2R) - P.s_r * Math.sin(alpha * D2R) + CFG.foot_offset, -1, 1)) * R2D;
  }
  function kneeAtPose(P, alpha, phi) {
    const a = alpha * D2R, p = phi * D2R;
    const K = [P.s_r * Math.sin(a), P.s_r * Math.cos(a)];
    const H = [K[0] - P.t_r * Math.cos(p), K[1] + P.t_r * Math.sin(p)];
    const u = [H[0] - K[0], H[1] - K[1]], v = [-K[0], -K[1]];
    return Math.acos(clamp((u[0] * v[0] + u[1] * v[1]) / (Math.hypot(...u) * Math.hypot(...v) + 1e-9), -1, 1)) * R2D;
  }
  function mlpPredict(M, x) {
    let a = x.map((v, i) => (v - M.mean[i]) / M.scale[i]);
    for (let l = 0; l < M.W.length; l++) {
      const W = M.W[l], b = M.b[l], out = new Array(b.length);
      for (let j = 0; j < b.length; j++) { let s = b[j]; for (let i = 0; i < a.length; i++) s += a[i] * W[i][j]; out[j] = s; }
      a = (l < M.W.length - 1) ? out.map(v => Math.max(0, v)) : out;
    }
    return 1 / (1 + Math.exp(-a[0]));
  }
  function featVec(f, P) { return [f.knee, f.hip, f.torso, f.shank, f.phi, P.femur_torso, P.shank_femur, P.leg_torso]; }
  function scoreRep(f, P) {
    const x = featVec(f, P);
    const reasons = [];
    if (f.knee > CFG.generic_knee_max) reasons.push('knee ' + f.knee.toFixed(0) + '° > ' + CFG.generic_knee_max + '°');
    if (f.torso > CFG.generic_torso_max) reasons.push('torso lean ' + f.torso.toFixed(0) + '° > ' + CFG.generic_torso_max + '°');
    if (f.phi > CFG.depth_phi_ok) reasons.push('hips above knee level');
    return { pers: mlpPredict(MODELS.personalized, x), gen: mlpPredict(MODELS.generic, x.slice(0, 5)),
             ruleOK: reasons.length === 0, ruleReasons: reasons };
  }
  // Geometry-aware coaching (uses the person's own proportions).
  function coach(f, P, phase) {
    const target = torsoNeeded(P, f.shank, f.phi), diff = f.torso - target, tol = CFG.torso_tol;
    const torso = Math.abs(diff) <= tol ? 'ok' : (diff > 0 ? 'forward' : 'upright');
    const msgs = [];
    if (torso === 'forward') msgs.push('Chest is folding forward more than YOUR proportions need (you ' + f.torso.toFixed(0) + '°, your balanced target ≈ ' + target.toFixed(0) + '°). Keep the chest up and let the knees travel.');
    if (torso === 'upright') msgs.push('Torso is more upright than your balance point (you ' + f.torso.toFixed(0) + '°, target ≈ ' + target.toFixed(0) + '°). Weight may drift to the heels: sit the hips back and hinge a little.');
    if (f.shank > CFG.alpha_max) msgs.push('Knees are travelling very far forward (' + f.shank.toFixed(0) + '°). Try a wider stance or a small heel lift.');
    const depthOK = f.phi <= CFG.depth_phi_ok;
    if (phase === 'rep' && !depthOK) msgs.push('Depth not reached: hips ended about ' + (f.phi - CFG.depth_phi_ok).toFixed(0) + '° above knee level.');
    return { target, diff, torso, depthOK, msgs };
  }
  // ====================================================================== PURE:END

  const $ = id => document.getElementById(id);
  const video = $('sq-video'), canvas = $('sq-canvas'), ctx = canvas.getContext('2d');
  const SIDES = { left: { sh: 11, hip: 23, knee: 25, ankle: 27, heel: 29, toe: 31 }, right: { sh: 12, hip: 24, knee: 26, ankle: 28, heel: 30, toe: 32 } };
  const CONN = [[11,12],[11,13],[13,15],[12,14],[14,16],[11,23],[12,24],[23,24],[23,25],[25,27],[27,29],[29,31],[27,31],[24,26],[26,28],[28,30],[30,32],[28,32],[15,17],[15,19],[15,21],[16,18],[16,20],[16,22],[0,1],[1,2],[2,3],[3,7],[0,4],[4,5],[5,6],[6,8],[9,10]];
  const GREEN = '#22c55e', ORANGE = '#f59e0b', RED = '#ef4444', CYAN = '#22d3ee', GREY = 'rgba(255,255,255,.45)';

  // mode: 'none' | 'camera' | 'file'
  const S = { lm: null, running: false, busy: false, loading: false, mode: 'none', stream: null, loopId: 0,
              calib: null, P: null, side: null, sideLocked: null, sm: {}, fwd: 1,
              state: 'standing', rep: null, nReps: 0, scores: [], log: [], rows: [], lastRow: null, lastVT: -1, frames: 0, t0: 0, fc: 0, seeking: false };
  window.__sqStop = () => { S.running = false; S.loopId++; if (S.stream) S.stream.getTracks().forEach(t => t.stop()); try { video.pause(); } catch (e) {} };

  const fmt = s => { s = Math.max(0, s || 0); return Math.floor(s / 60) + ':' + String(Math.floor(s % 60)).padStart(2, '0'); };
  function banner(msg, cls) { const b = $('sq-banner'); b.textContent = msg; b.style.color = cls === 'err' ? '#fca5a5' : cls === 'warn' ? '#fcd34d' : cls === 'ok' ? '#86efac' : '#e5e7eb'; }
  function setCalStatus(msg, cls) { const e = $('sq-calstatus'); e.textContent = msg; e.className = cls || 'mut'; }
  function ema(key, v, a) { a = a || 0.5; S.sm[key] = (S.sm[key] === undefined) ? v : a * v + (1 - a) * S.sm[key]; return S.sm[key]; }
  function clockNow() { return S.mode === 'file' ? video.currentTime : performance.now() / 1000; }
  function setStage(w, h) { if (!w || !h) return; canvas.width = w; canvas.height = h; $('sq-stage').style.aspectRatio = w + ' / ' + h; }
  function resetRepState() { S.state = 'standing'; S.rep = null; S.sm = {}; }
  function sideVis(lm, ids) { return Math.min(lm[ids.sh].visibility, lm[ids.hip].visibility, lm[ids.knee].visibility, lm[ids.ankle].visibility); }
  function bestSide(lm) {
    const m = ids => (lm[ids.sh].visibility + lm[ids.hip].visibility + lm[ids.knee].visibility + lm[ids.ankle].visibility) / 4;
    return m(SIDES.left) >= m(SIDES.right) ? 'left' : 'right';
  }

  // ================================================================= pose model
  async function loadLandmarker(variant) {
    S.loading = true; banner('Loading pose model (' + variant + ')…');
    try {
      if (S.lm) { try { S.lm.close(); } catch (e) {} S.lm = null; }
      const V = await import(CFG.vision_url);
      const files = await V.FilesetResolver.forVisionTasks(CFG.wasm_url);
      const make = delegate => V.PoseLandmarker.createFromOptions(files, {
        baseOptions: { modelAssetPath: CFG.model_urls[variant], delegate: delegate },
        runningMode: 'VIDEO', numPoses: 1, minPoseDetectionConfidence: 0.5, minPosePresenceConfidence: 0.5, minTrackingConfidence: 0.5 });
      try { S.lm = await make('GPU'); } catch (e) { S.lm = await make('CPU'); }
    } finally { S.loading = false; }
  }
  async function ensureLandmarker() { if (!S.lm) await loadLandmarker($('sq-model').value); }

  // ================================================================= sources (camera / uploaded video)
  function stopSource() {
    S.running = false; S.loopId++;
    if (S.stream) { S.stream.getTracks().forEach(t => t.stop()); S.stream = null; }
    try { video.pause(); } catch (e) {}
    video.srcObject = null; try { video.removeAttribute('src'); video.load(); } catch (e) {}
  }
  function startRun() {
    resetRepState(); S.lastVT = -1; S.frames = 0; S.t0 = performance.now(); S.running = true;
    $('sq-stop').disabled = false; $('sq-cal').disabled = S.mode !== 'camera';
    $('sq-vctl').style.display = S.mode === 'file' ? 'block' : 'none';
    const id = ++S.loopId;
    const tick = () => { if (!S.running || id !== S.loopId) return; requestAnimationFrame(tick); step(); };
    requestAnimationFrame(tick);
  }
  async function startCamera() {
    $('sq-start').disabled = true;
    try {
      await ensureLandmarker();
      stopSource();
      const stream = await navigator.mediaDevices.getUserMedia({ video: { width: { ideal: 960 }, height: { ideal: 540 }, facingMode: 'user' }, audio: false });
      S.stream = stream; video.srcObject = stream; await video.play();
      S.mode = 'camera'; setStage(video.videoWidth, video.videoHeight);
    } catch (e) {
      banner('Could not start the camera: ' + (e && e.message ? e.message : e) + '. Allow camera access, or upload a video instead.', 'err');
      $('sq-start').disabled = false; return;
    }
    startRun();
    banner(S.P ? 'Tracking. Squat when ready.' : 'Camera ready. Upload a standing photo (Step 1) or stand side-on and press the camera calibrate button.', S.P ? 'ok' : 'warn');
  }
  async function loadVideoFile(file) {
    if (!file) return;
    try {
      await ensureLandmarker();
      stopSource();
      S.mode = 'file';
      const url = URL.createObjectURL(file);
      video.muted = true; video.playsInline = true;
      await new Promise((res, rej) => {
        video.onloadedmetadata = res;
        video.onerror = () => rej(new Error('This video format cannot be decoded by your browser (try MP4/H.264 or WebM).'));
        video.src = url;
      });
      setStage(video.videoWidth, video.videoHeight);
      video.playbackRate = parseFloat($('sq-speed').value) || 1;
      resetSession(true);
      $('sq-start').disabled = false;
      await video.play();
      startRun();
      $('sq-play').textContent = 'Pause';
      banner(S.P ? 'Analysing "' + file.name + '" with your personalization…' : 'Analysing "' + file.name + '". Step 1 is not done yet: upload a standing photo, or press "Use the next 3 s of this video as my calibration" while the person stands.', S.P ? 'ok' : 'warn');
    } catch (e) { banner('Could not open the video: ' + (e && e.message ? e.message : e), 'err'); }
  }
  function onVideoEnded() {
    if (S.mode !== 'file') return;
    if (S.calib) { const c = S.calib; S.calib = null; finishCalibration(c, 'a segment of this video'); }
    if (S.state === 'down') { finishRep(); S.state = 'standing'; }
    $('sq-play').textContent = 'Replay';
    const n = S.nReps, avg = n ? Math.round(S.scores.reduce((a, c) => a + c, 0) / n) : 0, good = S.log.filter(r => r.score >= 50).length;
    banner(n ? 'Video finished: ' + n + ' rep' + (n > 1 ? 's' : '') + ', average score ' + avg + ', ' + good + ' judged good.' : 'Video finished: no complete squat repetitions were detected.', n ? 'ok' : 'warn');
  }
  function stopAll() {
    stopSource(); S.mode = 'none'; ctx.clearRect(0, 0, canvas.width, canvas.height);
    $('sq-start').disabled = false; $('sq-stop').disabled = true; $('sq-cal').disabled = true; $('sq-vctl').style.display = 'none'; banner('Stopped.');
  }

  // ================================================================= main per-frame step
  function step() {
    if (S.busy || S.loading || !S.lm || video.readyState < 2 || video.currentTime === S.lastVT) return;
    const vt = video.currentTime;
    if (S.mode === 'file' && S.lastVT >= 0 && Math.abs(vt - S.lastVT) > 0.5) resetRepState();   // seek/jump: don't let a rep span the gap
    S.lastVT = vt;
    if (video.videoWidth && canvas.width !== video.videoWidth) setStage(video.videoWidth, video.videoHeight);
    let res;
    try { res = S.lm.detectForVideo(video, performance.now()); } catch (e) { return; }
    S.frames++;
    const el = (performance.now() - S.t0) / 1000;
    if (el > 1) $('sq-fps').textContent = (S.frames / el).toFixed(0) + ' fps';
    if (S.mode === 'file' && !S.seeking && video.duration) { $('sq-seek').value = String(Math.round(vt / video.duration * 1000)); }
    if (S.mode === 'file') $('sq-time').textContent = fmt(vt) + ' / ' + fmt(video.duration);
    onResult(res);
  }

  function onResult(res) {
    const W = canvas.width, H = canvas.height;
    if (!res.landmarks || !res.landmarks.length) { ctx.clearRect(0, 0, W, H); banner('No person detected. Make sure the whole body is visible.', 'warn'); return; }
    const lm = res.landmarks[0], wl = res.worldLandmarks[0];
    let side = S.sideLocked || bestSide(lm);
    if (S.sideLocked && sideVis(lm, SIDES[side]) < 0.5) { const alt = bestSide(lm); if (sideVis(lm, SIDES[alt]) >= 0.5) side = alt; }
    const ids = SIDES[side];
    if (S.side !== side) { S.sm = {}; S.side = side; }
    const upGuess = S.P ? S.P.upSign : -1;
    const raw = featuresFromWorld(wl[ids.sh], wl[ids.hip], wl[ids.knee], wl[ids.ankle], upGuess);
    const f = { knee: ema('knee', raw.knee), hip: ema('hip', raw.hip), torso: ema('torso', raw.torso), shank: ema('shank', raw.shank), phi: ema('phi', raw.phi) };
    const fw = Math.sign(lm[ids.toe].x - lm[ids.heel].x) || 1; S.fwd = Math.sign(ema('fwd', fw, 0.1)) || 1;
    const info = S.P ? coach(f, S.P, 'live') : null;
    let status = { torso: null, depth: null, shank: null, target: null, inRep: false };

    if (sideVis(lm, ids) < 0.5) {
      banner('Your ' + side + ' side is not clearly visible. Turn side-on and keep hip, knee and ankle in view.', 'warn');
      drawSkeleton(lm, ids, status, f); return;
    }
    S.fc++; if (S.fc % 2 === 0 || S.calib) updateFeatures(lm, ids, f, info);
    if (S.calib) { runCalibration(raw, wl, lm, ids, side); drawSkeleton(lm, ids, status, f); return; }
    if (!S.P) {
      banner(S.mode === 'file' ? 'Not personalized: upload a standing photo (Step 1) or use the next 3 s of this video as calibration.' : 'Not personalized: upload a standing photo (Step 1) or press the camera calibrate button.', 'warn');
      drawSkeleton(lm, ids, status, f); return;
    }

    // ----- live geometry-aware feedback + rep detection
    const inRep = f.knee < CFG.rep_start_knee;
    status = { torso: inRep ? info.torso : null, depth: f.phi <= CFG.depth_phi_ok, shank: f.shank > CFG.alpha_max ? 'bad' : 'ok', target: info.target, inRep: inRep };
    if (S.state === 'standing' && inRep) { S.state = 'down'; S.rep = { minPhi: 999, minKnee: 999, bottom: null }; }
    if (S.state === 'down') {
      S.rep.minKnee = Math.min(S.rep.minKnee, f.knee);
      if (f.phi < S.rep.minPhi) { S.rep.minPhi = f.phi; S.rep.bottom = Object.assign({}, f); }
      if (f.knee > CFG.rep_end_knee) { finishRep(); S.state = 'standing'; }
    }
    updateLive(f, info, inRep);
    drawSkeleton(lm, ids, status, f);
  }

  // ================================================================= calibration (camera / video segment / uploaded photos & clips)
  function newCalib(mode) { return { mode: mode, t0: clockNow(), thigh: [], shank: [], torso: [], dy: [], sides: [], hy: [], t2d: [] }; }
  function addSample(c, raw, wl, lm, ids, side, ar, relaxed) {
    if (!(raw.knee > 150 && raw.torso < (relaxed ? 30 : 25))) return false;     // must be a standing pose
    c.thigh.push(raw.thighLen); c.shank.push(raw.shankLen); c.torso.push(raw.torsoLen);
    c.dy.push(wl[ids.hip].y - wl[ids.knee].y); c.sides.push(side);
    c.hy.push(lm[ids.hip].y); c.t2d.push(Math.hypot((lm[ids.hip].x - lm[ids.knee].x) * ar, lm[ids.hip].y - lm[ids.knee].y));
    return true;
  }
  function startCalibration() {
    if (!S.running) return;
    S.P = null; S.sideLocked = null; resetRepState();
    S.calib = newCalib(S.mode === 'file' ? 'video' : 'camera');
    if (S.mode === 'file' && video.paused) video.play();
    setCalStatus('Calibrating from ' + (S.mode === 'file' ? 'the video' : 'the camera') + '…', 'warn'); $('sq-profile').textContent = 'Calibrating…';
  }
  function runCalibration(raw, wl, lm, ids, side) {
    const c = S.calib, warm = c.mode === 'camera' ? 1 : 0, el = clockNow() - c.t0, total = warm + CFG.calib_seconds;
    if (el < warm) { banner('Stand tall, side-on, arms relaxed. Starting in ' + (warm - el).toFixed(1) + ' s…', 'warn'); return; }
    if (el < total) {
      banner('Hold still… ' + (total - el).toFixed(1) + ' s  (standing samples: ' + c.thigh.length + ')', 'warn');
      addSample(c, raw, wl, lm, ids, side, canvas.width / canvas.height, false); return;
    }
    S.calib = null; finishCalibration(c, c.mode === 'camera' ? 'the live camera' : 'a segment of this video');
  }
  function finishCalibration(c, source) {
    const minN = c.mode === 'files' ? 1 : 15;
    if (c.thigh.length < minN) {
      const msg = c.mode === 'files' ? 'No usable standing frame found. Use a side-on, full-body, standing photo (knees straight, whole body incl. feet visible).' : 'Not enough clean standing frames (' + c.thigh.length + '). Stand tall, side-on, full body in view, then calibrate again.';
      setCalStatus(msg, 'err'); banner(msg, 'err'); $('sq-profile').textContent = 'Calibration failed.'; return false;
    }
    const up = Math.sign(median(c.dy)) || -1;
    S.P = profileFromLengths(median(c.thigh), median(c.shank), median(c.torso), up);
    S.P.hipY0 = median(c.hy); S.P.thigh2d = median(c.t2d); S.P.source = source; S.P.n = c.thigh.length;
    S.sideLocked = c.sides.filter(s => s === 'left').length >= c.sides.length / 2 ? 'left' : 'right';
    resetRepState(); showProfile();
    setCalStatus('Personalized from ' + source + ' (' + c.thigh.length + ' clean standing frame' + (c.thigh.length > 1 ? 's' : '') + '). Now start the camera or upload a squat video.', 'ok');
    banner('Personalized. Now squat (live) or upload a squat video: the skeleton turns green when your form matches YOUR geometry.', 'ok');
    return true;
  }
  function showProfile() {
    const P = S.P, lo = CFG.alpha_range[0], hi = CFG.alpha_range[1];
    const tLo = torsoNeeded(P, hi, 0), tHi = torsoNeeded(P, lo, 0), kLo = kneeAtPose(P, hi, 0), kHi = kneeAtPose(P, lo, 0);
    $('sq-profile').innerHTML =
      '<table><tr><td class="k">Femur / torso</td><td class="v"><b>' + P.femur_torso.toFixed(2) + '</b></td><td class="k">Shank / femur</td><td class="v"><b>' + P.shank_femur.toFixed(2) + '</b></td></tr>' +
      '<tr><td class="k">Leg / torso</td><td class="v"><b>' + P.leg_torso.toFixed(2) + '</b></td><td class="k">Tracked side</td><td class="v"><b>' + S.sideLocked + '</b></td></tr></table>' +
      '<div class="mut" style="margin-top:4px">Source: ' + P.source + '</div>' +
      '<div style="margin-top:6px">At parallel depth <b>your</b> balanced torso lean is about <b class="ok">' + tLo.toFixed(0) + '&deg;&ndash;' + tHi.toFixed(0) + '&deg;</b> ' +
      '(a fixed rule would allow &le; ' + CFG.generic_torso_max + '&deg; for everyone) and knee angle about <b class="ok">' + kLo.toFixed(0) + '&deg;&ndash;' + kHi.toFixed(0) + '&deg;</b>.</div>';
  }

  // ---- photos / clips for personalization
  function toCanvas(src, w, h) {
    const k = Math.min(1, CFG.calib_max_px / Math.max(w, h)), cw = Math.round(w * k), chh = Math.round(h * k);
    const c = document.createElement('canvas'); c.width = cw; c.height = chh; c.getContext('2d').drawImage(src, 0, 0, cw, chh); return c;
  }
  async function framesFromFile(file) {
    const url = URL.createObjectURL(file);
    if ((file.type || '').startsWith('image/')) {
      const img = new Image(); img.src = url; await img.decode();
      return [toCanvas(img, img.naturalWidth, img.naturalHeight)];
    }
    const v = document.createElement('video'); v.muted = true; v.playsInline = true; v.preload = 'auto';
    await new Promise((res, rej) => { v.onloadedmetadata = res; v.onerror = () => rej(new Error('cannot decode ' + file.name)); v.src = url; });
    const dur = Math.min(isFinite(v.duration) ? v.duration : CFG.calib_clip_max_s, CFG.calib_clip_max_s);
    const n = clamp(Math.floor(dur * 5), 4, CFG.calib_max_frames), out = [];
    for (let i = 0; i < n; i++) {
      await new Promise(res => { v.onseeked = res; v.currentTime = dur * (i + 0.5) / n; });
      out.push(toCanvas(v, v.videoWidth, v.videoHeight));
    }
    return out;
  }
  function showStill(cv, lm, ids, f) {
    setStage(cv.width, cv.height); ctx.clearRect(0, 0, canvas.width, canvas.height); ctx.drawImage(cv, 0, 0, canvas.width, canvas.height);
    drawSkeleton(lm, ids, { torso: null, depth: null, shank: null, target: null, inRep: false }, f, true);
  }
  async function personalizeFromFiles(fileList) {
    const files = Array.from(fileList || []); if (!files.length) return;
    S.busy = true; setCalStatus('Reading ' + files.length + ' file(s)…', 'warn'); banner('Extracting frames and measuring body proportions…', 'warn');
    let imageMode = false;
    try {
      await ensureLandmarker();
      const frames = [], skipped = [];
      for (const f of files) { try { frames.push(...await framesFromFile(f)); } catch (e) { skipped.push(f.name); } }
      if (!frames.length) throw new Error('None of the files could be read (' + skipped.join(', ') + '). Use JPG/PNG photos or MP4/WebM clips.');
      await S.lm.setOptions({ runningMode: 'IMAGE' }); imageMode = true;
      const c = newCalib('files'); let best = null, seen = 0;
      for (const cv of frames) {
        const res = S.lm.detect(cv);
        if (!res.landmarks || !res.landmarks.length) continue;
        const lm = res.landmarks[0], wl = res.worldLandmarks[0], side = bestSide(lm), ids = SIDES[side];
        if (sideVis(lm, ids) < 0.5) continue;
        seen++;
        const raw = featuresFromWorld(wl[ids.sh], wl[ids.hip], wl[ids.knee], wl[ids.ankle], -1);
        if (addSample(c, raw, wl, lm, ids, side, cv.width / cv.height, true) && !best) best = { cv: cv, lm: lm, ids: ids, f: { knee: raw.knee, hip: raw.hip, torso: raw.torso, shank: raw.shank, phi: raw.phi } };
      }
      await S.lm.setOptions({ runningMode: 'VIDEO' }); imageMode = false;
      S.mode = (S.mode === 'file' || S.mode === 'camera') ? S.mode : 'none';
      const ok = finishCalibration(c, files.length + ' file(s), ' + frames.length + ' frame(s) analysed');
      if (best && !S.running) showStill(best.cv, best.lm, best.ids, best.f);
      if (!ok && seen === 0) setCalStatus('No person with a clearly visible side was found in the uploaded file(s).', 'err');
    } catch (e) {
      setCalStatus('Personalization failed: ' + (e && e.message ? e.message : e), 'err'); banner('Personalization failed: ' + (e && e.message ? e.message : e), 'err');
    } finally {
      if (imageMode) { try { await S.lm.setOptions({ runningMode: 'VIDEO' }); } catch (e) {} }
      S.busy = false; $('sq-calfile').value = '';
    }
  }

  // ================================================================= reps
  function finishRep() {
    const r = S.rep; S.rep = null;
    if (!r || !r.bottom || r.minKnee > CFG.min_rep_knee) { return; }   // too shallow to count as a rep attempt
    const b = r.bottom, sc = scoreRep(b, S.P), info = coach(b, S.P, 'rep');
    S.nReps++; S.scores.push(sc.pers * 100);
    const pct = Math.round(sc.pers * 100), ok = sc.pers >= 0.5;
    S.log.push({ n: S.nReps, t: fmt(S.mode === 'file' ? video.currentTime : (performance.now() - S.t0) / 1000), score: pct, knee: Math.round(b.knee), torso: Math.round(b.torso) });
    $('sq-repno').textContent = S.nReps;
    $('sq-score').textContent = pct; $('sq-score').className = 'big ' + (pct >= 70 ? 'ok' : pct >= 50 ? 'warn' : 'err');
    $('sq-verdict').innerHTML = ok ? (pct >= 70 ? '<b class="ok">Good form for your body</b>' : '<b class="warn">Borderline</b>') : '<b class="err">Needs correction</b>';
    $('sq-scorebar').style.width = pct + '%'; $('sq-scorebar').style.background = pct >= 70 ? GREEN : pct >= 50 ? ORANGE : RED;
    const fix = info.msgs.length ? info.msgs.map(m => '&bull; ' + m).join('<br>') : '<span class="ok">&bull; Depth, balance and knee travel all match your proportions.</span>';
    $('sq-repdetail').innerHTML = fix +
      '<div style="margin-top:6px;border-top:1px solid #334155;padding-top:6px">Same rep, other systems:<br>' +
      'Personalized ML: <b>' + pct + '</b> ' + (ok ? '&#10003;' : '&#10007;') + ' &nbsp;|&nbsp; Generic ML (angles only): <b>' + Math.round(sc.gen * 100) + '</b> ' + (sc.gen >= .5 ? '&#10003;' : '&#10007;') +
      '<br>Fixed rule: ' + (sc.ruleOK ? '<b class="ok">&#10003; pass</b>' : '<b class="err">&#10007; fail</b> <span class="mut">(' + sc.ruleReasons.join(', ') + ')</span>') + '</div>';
    $('sq-avg').textContent = Math.round(S.scores.reduce((a, c) => a + c, 0) / S.scores.length);
    renderLog();
    const row = { person_id: $('sq-pid').value || 'p01', rep_id: S.nReps, knee: b.knee, hip: b.hip, torso: b.torso, shank: b.shank, phi: b.phi,
                  femur_torso: S.P.femur_torso, shank_femur: S.P.shank_femur, leg_torso: S.P.leg_torso, label: '' };
    S.rows.push(row); S.lastRow = row; ['sq-good', 'sq-bad'].forEach(i => $(i).disabled = false);
    banner('Rep ' + S.nReps + ' scored: ' + pct + '/100', ok ? 'ok' : 'warn');
  }
  function renderLog() {
    const n = S.log.length, good = S.log.filter(r => r.score >= 50).length;
    $('sq-sum').textContent = n ? n + ' rep' + (n > 1 ? 's' : '') + ' · average ' + Math.round(S.scores.reduce((a, c) => a + c, 0) / n) + ' · ' + good + ' judged good' : '';
    const rows = S.log.slice(-12).reverse().map(r => '<tr><td>#' + r.n + '</td><td class="mut">' + r.t + '</td><td class="' + (r.score >= 70 ? 'ok' : r.score >= 50 ? 'warn' : 'err') + '"><b>' + r.score + '</b></td><td class="mut">knee ' + r.knee + '° · torso ' + r.torso + '°</td></tr>').join('');
    $('sq-log').innerHTML = rows ? '<table>' + rows + '</table>' : 'No reps yet.';
  }
  function label(v) { if (!S.lastRow) return; S.lastRow.label = v; S.lastRow.person_id = $('sq-pid').value || S.lastRow.person_id; refreshCSV(); banner('Rep ' + S.lastRow.rep_id + ' labelled ' + (v ? 'GOOD' : 'FAULTY'), 'ok'); }
  function csvText() {
    const cols = ['person_id', 'rep_id', 'knee', 'hip', 'torso', 'shank', 'phi', 'femur_torso', 'shank_femur', 'leg_torso', 'label'];
    const rows = S.rows.filter(r => r.label !== '');
    return [cols.join(',')].concat(rows.map(r => cols.map(c => (typeof r[c] === 'number' && c !== 'rep_id' && c !== 'label') ? r[c].toFixed(4) : r[c]).join(','))).join('\n');
  }
  function refreshCSV() { const n = S.rows.filter(r => r.label !== '').length; $('sq-nrows').textContent = n; $('sq-csv').value = csvText(); $('sq-dl').disabled = n === 0; }
  function download() {
    const a = document.createElement('a'); a.href = URL.createObjectURL(new Blob([csvText()], { type: 'text/csv' })); a.download = 'squat_labelled_reps.csv'; document.body.appendChild(a); a.click(); a.remove();
  }
  function resetSession(keepCalib) {
    S.calib = null; resetRepState(); S.nReps = 0; S.scores = []; S.log = []; S.lastVT = -1;
    ['sq-score', 'sq-avg'].forEach(i => $(i).textContent = '–'); $('sq-score').className = 'big'; $('sq-repno').textContent = '0'; $('sq-verdict').innerHTML = ''; $('sq-scorebar').style.width = '0';
    $('sq-repdetail').textContent = 'Do a squat (or play a video) to get a score.'; $('sq-live').textContent = '—'; $('sq-sum').textContent = ''; $('sq-log').textContent = 'No reps yet.';
    if (!keepCalib) { S.P = null; S.sideLocked = null; $('sq-profile').textContent = 'Not calibrated yet.'; setCalStatus('Not personalized yet — upload a standing photo to begin.', 'warn'); }
    banner(keepCalib ? 'Session reset (personalization kept).' : 'Session reset. Personalize again (Step 1).');
  }

  // ================================================================= UI updates
  function updateLive(f, info, inRep) {
    const depthPct = clamp((90 - f.phi) / (90 - CFG.depth_phi_ok), 0, 1);
    $('sq-depthbar').style.width = (depthPct * 100) + '%'; $('sq-depthbar').style.background = f.phi <= CFG.depth_phi_ok ? GREEN : ORANGE;
    $('sq-depthtxt').textContent = f.phi <= CFG.depth_phi_ok ? 'Depth reached: hips at or below knee level' : 'Hips are ' + (f.phi - CFG.depth_phi_ok).toFixed(0) + '° above the depth line';
    if (!inRep) { $('sq-live').innerHTML = '<span class="mut">Standing. Squat when ready.</span>'; return; }
    const live = info.msgs.filter(m => !m.startsWith('Depth'));
    $('sq-live').innerHTML = live.length ? live.map(m => '<span class="warn">&bull; ' + m + '</span>').join('<br>') : '<span class="ok">&bull; Torso balance matches your proportions.</span>';
  }
  function updateFeatures(lm, ids, f, info) {
    const P = S.P, dash = '<span class="mut">&mdash;</span>', c3 = i => lm[i].x.toFixed(3) + ', ' + lm[i].y.toFixed(3);
    const hdr = P ? (lm[ids.hip].y - P.hipY0) / (P.thigh2d || 1) : null;
    const row = (k, v, t) => '<tr><td class="k">' + k + '</td><td class="v">' + v + '</td><td class="mut">' + (t || '') + '</td></tr>';
    const deg = x => x.toFixed(1) + '°';
    let h = '<table>';
    h += row('shoulder (x, y)', c3(ids.sh)) + row('hip (x, y)', c3(ids.hip)) + row('knee (x, y)', c3(ids.knee)) + row('ankle (x, y)', c3(ids.ankle));
    h += '<tr class="sep"><td colspan="3" class="mut">Joint kinematics &rarr; model inputs</td></tr>';
    h += row('knee_angle', deg(f.knee), P ? 'hip-knee-ankle' : '') + row('hip_angle', deg(f.hip), 'shoulder-hip-knee');
    h += row('torso_lean', deg(f.torso), info ? 'your target ≈ ' + info.target.toFixed(0) + '°' : 'from vertical') + row('shin_lean', deg(f.shank), 'ankle-knee vs vertical');
    h += row('thigh_elevation', P ? deg(f.phi) : dash, P ? 'depth reached at ≤ ' + CFG.depth_phi_ok + '°' : 'needs personalization');
    h += row('hip_depth_ratio', hdr === null ? dash : hdr.toFixed(2), hdr === null ? 'needs personalization' : 'hip drop / thigh length (display only)');
    h += '<tr class="sep"><td colspan="3" class="mut">Static body ratios &rarr; model inputs</td></tr>';
    h += row('femur_torso', P ? P.femur_torso.toFixed(3) : dash) + row('shank_femur', P ? P.shank_femur.toFixed(3) : dash) + row('leg_torso', P ? P.leg_torso.toFixed(3) : dash);
    $('sq-feat').innerHTML = h + '</table>';
  }

  // ================================================================= drawing
  function drawSkeleton(lm, ids, st, f, keepBg) {
    const W = canvas.width, H = canvas.height; if (!keepBg) ctx.clearRect(0, 0, W, H);
    const P = i => [lm[i].x * W, lm[i].y * H];
    ctx.lineCap = 'round'; ctx.lineWidth = Math.max(2, W / 400); ctx.strokeStyle = GREY;
    for (const [a, b] of CONN) { if (lm[a].visibility < 0.3 || lm[b].visibility < 0.3) continue; const pa = P(a), pb = P(b); ctx.beginPath(); ctx.moveTo(pa[0], pa[1]); ctx.lineTo(pb[0], pb[1]); ctx.stroke(); }
    ctx.fillStyle = 'rgba(255,255,255,.7)';
    for (let i = 0; i < lm.length; i++) { if (lm[i].visibility < 0.3) continue; const p = P(i); ctx.beginPath(); ctx.arc(p[0], p[1], W / 300 + 1, 0, 7); ctx.fill(); }

    const sh = P(ids.sh), hip = P(ids.hip), kn = P(ids.knee), an = P(ids.ankle);
    const col = { torso: st.torso === 'ok' ? GREEN : st.torso === 'forward' || st.torso === 'upright' ? RED : CYAN,
                  thigh: st.inRep ? (st.depth ? GREEN : ORANGE) : CYAN, shank: st.shank === 'bad' ? RED : (st.inRep ? GREEN : CYAN) };
    const seg = (a, b, c) => { ctx.strokeStyle = c; ctx.lineWidth = Math.max(4, W / 160); ctx.beginPath(); ctx.moveTo(a[0], a[1]); ctx.lineTo(b[0], b[1]); ctx.stroke(); };
    seg(sh, hip, col.torso); seg(hip, kn, col.thigh); seg(kn, an, col.shank);
    for (const p of [sh, hip, kn, an]) { ctx.fillStyle = '#fff'; ctx.beginPath(); ctx.arc(p[0], p[1], Math.max(5, W / 130), 0, 7); ctx.fill(); }

    // ghost: where YOUR torso should be for your proportions at this depth
    if (st.target !== null && st.target !== undefined && st.inRep) {
      const L = Math.hypot(sh[0] - hip[0], sh[1] - hip[1]), t = st.target * D2R;
      ctx.setLineDash([10, 8]); ctx.strokeStyle = CYAN; ctx.lineWidth = Math.max(3, W / 200);
      ctx.beginPath(); ctx.moveTo(hip[0], hip[1]); ctx.lineTo(hip[0] + S.fwd * L * Math.sin(t), hip[1] - L * Math.cos(t)); ctx.stroke(); ctx.setLineDash([]);
    }
    if (f) {
      tag(kn[0] + 12, kn[1], 'knee ' + f.knee.toFixed(0) + '°'); tag(hip[0] + 12, hip[1] - 4, 'hip ' + f.hip.toFixed(0) + '°');
      tag(sh[0] + 12, sh[1], 'torso ' + f.torso.toFixed(0) + '°');
    }
  }
  function tag(x, y, text) {
    ctx.font = 'bold ' + Math.max(12, canvas.width / 55) + 'px system-ui,sans-serif'; const w = ctx.measureText(text).width + 8, h = Math.max(16, canvas.width / 40);
    x = Math.min(x, canvas.width - w - 2);
    ctx.fillStyle = 'rgba(15,23,42,.75)'; ctx.fillRect(x, y - h + 4, w, h); ctx.fillStyle = '#fff'; ctx.fillText(text, x + 4, y);
  }

  // ================================================================= wiring
  $('sq-start').onclick = startCamera;
  $('sq-cal').onclick = startCalibration; $('sq-vcal').onclick = startCalibration;
  $('sq-stop').onclick = stopAll;
  $('sq-reset').onclick = () => resetSession(false);
  $('sq-calfile').onchange = e => personalizeFromFiles(e.target.files);
  $('sq-vidfile').onchange = e => { const f = e.target.files && e.target.files[0]; loadVideoFile(f); e.target.value = ''; };
  $('sq-play').onclick = () => {
    if (video.ended || (video.duration && video.currentTime >= video.duration - 0.05)) { resetSession(true); video.currentTime = 0; video.play(); $('sq-play').textContent = 'Pause'; return; }
    if (video.paused) { video.play(); $('sq-play').textContent = 'Pause'; } else { video.pause(); $('sq-play').textContent = 'Play'; }
  };
  $('sq-speed').onchange = () => { video.playbackRate = parseFloat($('sq-speed').value) || 1; };
  $('sq-seek').oninput = () => { S.seeking = true; if (video.duration) video.currentTime = $('sq-seek').value / 1000 * video.duration; };
  $('sq-seek').onchange = () => { S.seeking = false; };
  video.onended = onVideoEnded;
  $('sq-good').onclick = () => label(1); $('sq-bad').onclick = () => label(0); $('sq-dl').onclick = download;
  $('sq-model').onchange = async () => { if (S.lm) { S.busy = true; try { await loadLandmarker($('sq-model').value); banner('Pose model switched.', 'ok'); } finally { S.busy = false; } } };
  window.__sqTest = { featuresFromWorld, profileFromLengths, torsoNeeded, kneeAtPose, mlpPredict, featVec, scoreRep, coach, onResult, startCalibration, step,
                      personalizeFromFiles, loadVideoFile, onVideoEnded, resetSession, S };
})();
</script>
"""

MP_VER = "0.10.14"
JS_CFG = dict(CFG)
JS_CFG.update(
    vision_url = f"https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@{MP_VER}/vision_bundle.mjs",
    wasm_url   = f"https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@{MP_VER}/wasm",
    model_urls = {v: f"https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_{v}/float16/latest/pose_landmarker_{v}.task"
                  for v in ("lite", "full", "heavy")},
)
# the trained models + every threshold are injected, so the browser runs exactly what you trained above
app_html = APP_TEMPLATE.replace("__CFG__", json.dumps(JS_CFG)).replace("__MODELS__", json.dumps(MODELS))
with open("squat_coach_live.html", "w", encoding="utf-8") as fh:
    fh.write(app_html)

display(HTML(app_html))
print("Saved a standalone copy: squat_coach_live.html  (see the optional cell below to download it)")

In [ ]:
#@title (Optional) Download the standalone app to run on your own computer { display-mode: "form" }
# Then on your computer:  python -m http.server 8000   and open  http://localhost:8000/squat_coach_live.html
from google.colab import files
files.download("squat_coach_live.html")

## 7 · Turning this into real evidence (what to do next)

1. **Collect labelled reps with the live app.** After each rep, a coach presses *GOOD* or *FAULTY*; use a unique **Participant ID** per person; *Download CSV*. Columns: `person_id, rep_id, knee, hip, torso, shank, phi, femur_torso, shank_femur, leg_torso, label`.
2. **Recruit for body diversity** (femur/torso and leg/torso spread, height, sex, ankle mobility), ideally ≥ 30 people × ≥ 10 reps, **two independent coaches** per rep (report inter-rater agreement).
3. **Keep the camera protocol fixed**: side-on, level at hip height, 2–3 m, full body incl. feet, plain lighting.
4. Set `USE_REAL_DATA = True` in section 2, upload the CSV(s), and re-run sections 2–5. The audit and bootstrap then become your real result.
5. Report: accuracy and **FRR/FAR per body-type group**, the **gap with 95 % bootstrap CIs** versus both baselines, the **ratio-shuffle check**, and the **noise sensitivity**.
6. Extensions: front-view calibration for hip/shoulder width, a push-up module (same pipeline, new geometry), fitting the balance-model constants (`foot_offset`, tolerance) from real data.